In [4]:
# -*- coding: utf-8 -*-
# Import necessary libraries
import sympy
from sympy import symbols, Matrix, Function, sin, exp, diff, latex, simplify
from IPython.display import display, Math

try:
    from gr import (
        compute_christoffel_symbols, display_christoffel_symbols,
        compute_riemann_curvature_tensor, display_riemann_curvature_tensor,
        compute_ricci_tensor, display_ricci_tensor,
        compute_ricci_scalar, display_ricci_scalar,
        compute_einstein_tensor, display_einstein_tensor
    )
    print("Successfully imported functions from 'gr' module.")
except ModuleNotFoundError:
    !git clone https://github.com/ToelUl/Einstein-tensor-calculator.git
    !cp -r Einstein-tensor-calculator/gr ./
    from gr import (
        compute_christoffel_symbols, display_christoffel_symbols,
        compute_riemann_curvature_tensor, display_riemann_curvature_tensor,
        compute_ricci_tensor, display_ricci_tensor,
        compute_ricci_scalar, display_ricci_scalar,
        compute_einstein_tensor, display_einstein_tensor
    )

Successfully imported functions from 'gr' module.


# Tutorial: Analyzing the Kerr Metric with `gr`

This notebook tackles the Kerr metric, a fundamental solution in General Relativity describing the spacetime geometry around a **rotating, uncharged, axisymmetric black hole** (or rotating star/object in vacuum). We will use the `gr` Python module to define the metric in Boyer-Lindquist coordinates and compute its geometric tensors.

**WARNING:** The algebraic complexity of the Kerr metric is very high. Calculating tensors like the Riemann and Ricci tensors, and especially verifying that the Ricci tensor is zero, can be **extremely computationally intensive** for symbolic systems. These calculations may take a very long time (potentially minutes or more) and require significant memory. Successful simplification to zero depends heavily on the capabilities of SymPy's `simplify` function used within the `gr` module.

## 1. Introduction: The Kerr Metric

The Kerr metric is the unique asymptotically flat, stationary, and axisymmetric solution to the Einstein Field Equations in vacuum ($R_{\mu\nu} = 0$). It is characterized by two parameters:
* $M$: The mass of the rotating object.
* $a$: The angular momentum per unit mass ($a = J/M$). We assume $|a| \le M$ for the existence of an event horizon (otherwise it describes a "naked singularity").

We will use **Boyer-Lindquist coordinates** $(t, r, \theta, \phi)$, which are adapted to the spacetime's symmetries but can be ill-behaved near the singularity. In these coordinates, the metric depends on auxiliary functions:

* $\Sigma(r, \theta) = r^2 + a^2 \cos^2\theta$
* $\Delta(r) = r^2 - 2Mr + a^2$

The Kerr line element is given by:

$ds^2 = -\left(1 - \frac{2Mr}{\Sigma}\right) dt^2 - \frac{4aMr\sin^2\theta}{\Sigma} dt d\phi + \frac{\Sigma}{\Delta} dr^2 + \Sigma d\theta^2 + \left(r^2 + a^2 + \frac{2a^2 Mr\sin^2\theta}{\Sigma}\right)\sin^2\theta d\phi^2$

**Key Features:**
* **Off-Diagonal Term ($g_{t\phi}$):** The term coupling $dt$ and $d\phi$ is non-zero ($g_{t\phi} = g_{\phi t} = -2aMr\sin^2\theta / \Sigma$). This term is responsible for **frame-dragging** (the Lense-Thirring effect), where spacetime itself is "dragged" around by the rotation of the mass.
* **Singularity:** The curvature singularity is not a point but a **ring** located at $r=0$ and $\theta=\pi/2$ (in the equatorial plane).
* **Event Horizons:** Given by the roots of $\Delta(r) = 0$, i.e., $r^2 - 2Mr + a^2 = 0$. The solutions are $r_\pm = M \pm \sqrt{M^2 - a^2}$. $r_+$ is the outer event horizon, and $r_-$ is the inner Cauchy horizon. They exist only if $|a| \le M$.
* **Ergosphere:** A region bounded by the outer event horizon ($r=r_+$) and the **static limit surface**, defined by $g_{tt} = 0$. The static limit occurs at $r = M + \sqrt{M^2 - a^2 \cos^2\theta}$. Inside the ergosphere (but outside $r_+$), observers *must* co-rotate with the black hole due to extreme frame-dragging. Energy can be extracted from this region (Penrose process).
* **Vacuum Solution:** Like Schwarzschild, the Kerr metric describes the vacuum spacetime outside the rotating source. Therefore, we expect its **Ricci tensor ($R_{\mu\nu}$), Ricci scalar ($R$), and Einstein tensor ($G_{\mu\nu}$) to be identically zero**. Verifying this is a crucial check.
* **Curvature:** The spacetime is curved, indicated by a non-zero Riemann tensor ($R^\rho{}_{\sigma\mu\nu}$).

Let's define the metric and explore these features.

In [5]:
# --- Define Coordinates, Parameters, and Metric ---

# Define Boyer-Lindquist coordinates (t, r, theta, phi)
t, r, theta, phi = symbols("t r θ φ", real=True)
coords = [t, r, theta, phi]
print("Coordinates:")
display(coords)

# Define the mass M and spin parameter a (real constants)
# Assume M > 0. Often |a| <= M is assumed for black holes.
M, a = symbols("M a", real=True)
print("\nParameters:")
display(M, a)

# Define auxiliary functions Sigma and Delta
Sigma = r**2 + a**2 * cos(theta)**2
Delta = r**2 - 2*M*r + a**2
print("\nAuxiliary Functions:")
display(Math(f"Σ = {latex(Sigma)}"))
display(Math(f"Δ = {latex(Delta)}"))

# Define the metric tensor components for Kerr metric
g_tt = -(1 - 2*M*r / Sigma)
g_tphi = - (2 * a * M * r * sin(theta)**2) / Sigma
g_rr = Sigma / Delta
g_thetatheta = Sigma
g_phiphi = (r**2 + a**2 + (2 * a**2 * M * r * sin(theta)**2) / Sigma) * sin(theta)**2

# Construct the 4x4 metric tensor as a SymPy Matrix
# Remember g_μν is symmetric, so g_phit = g_tphi
metric_kerr = Matrix([
    [ g_tt,    0,    0,    g_tphi      ],
    [  0,    g_rr,   0,       0        ],
    [  0,      0, g_thetatheta, 0      ],
    [ g_tphi, 0,    0,    g_phiphi    ]
])

print("\nKerr Metric Tensor g_μν:")
# Display the metric using LaTeX
# Use simplify for slightly cleaner display of g_phiphi if possible
g_phiphi_simplified = simplify(g_phiphi)
metric_kerr_simplified_display = Matrix([
    [ g_tt,    0,    0,    g_tphi      ],
    [  0,    g_rr,   0,       0        ],
    [  0,      0, g_thetatheta, 0      ],
    [ g_tphi, 0,    0,    g_phiphi_simplified    ]
])
display(Math(latex(metric_kerr_simplified_display)))

# Display components (optional, can be long)
# print("\nMetric Components:")
# display(Math(f"g_{{tt}} = {latex(g_tt)}"))
# display(Math(f"g_{{rr}} = {latex(g_rr)}"))
# display(Math(f"g_{{θθ}} = {latex(g_thetatheta)}"))
# display(Math(f"g_{{φφ}} = {latex(g_phiphi_simplified)}"))
# display(Math(f"g_{{tφ}} = {latex(g_tphi)}"))

# --- Calculate Horizon Locations ---
print("\nCalculating Horizon Locations (roots of Δ = 0):")
try:
    horizon_eq = Delta
    r_horizons = solve(horizon_eq, r)
    if r_horizons:
        display(Math(f"r_± = {latex(r_horizons)}"))
    else:
        print("Could not solve for horizons symbolically.")
except Exception as e:
    print(f"Error solving for horizons: {e}")

# --- Calculate Static Limit ---
print("\nCalculating Static Limit (outer boundary of ergosphere, where g_tt = 0):")
try:
    # g_tt = -(1 - 2*M*r / Sigma) = 0  => Sigma = 2*M*r
    # r^2 + a^2*cos(theta)^2 = 2*M*r
    static_limit_eq = Sigma - 2*M*r
    r_static_limit = solve(static_limit_eq, r)
    if r_static_limit:
        display(Math(f"r_{{static}} = {latex(r_static_limit)}")) # Should yield M + sqrt(M^2 - a^2*cos^2(theta))
        # Let's simplify the known solution to verify
        known_static = M + sqrt(M**2 - a**2*cos(theta)**2)
        if simplify(r_static_limit[1] - known_static) == 0: # Sympy might return +/- solution
             print("(Matches expected form M + sqrt(M^2 - a^2*cos^2(theta)))")
        else:
             print("(Symbolic result may differ slightly from expected form)")
    else:
        print("Could not solve for static limit symbolically.")
except Exception as e:
    print(f"Error solving for static limit: {e}")

Coordinates:


[t, r, θ, φ]


Parameters:


M

a

...


Calculating Horizon Locations (roots of Δ = 0):


<IPython.core.display.Math object>


Calculating Static Limit (outer boundary of ergosphere, where g_tt = 0):


<IPython.core.display.Math object>

(Matches expected form M + sqrt(M^2 - a^2*cos^2(theta)))


## 2. Christoffel Symbols ($Γ^ρ_{μν}$)

The Christoffel symbols for the Kerr metric are significantly more complex than for Schwarzschild due to the rotation ($a \neq 0$) and the $g_{t\phi}$ term. They encode the complicated inertial forces in the rotating, curved spacetime.

$\Gamma^\rho_{\mu\nu} = \frac{1}{2} g^{\rho\sigma} \left( \frac{\partial g_{\nu\sigma}}{\partial x^\mu} + \frac{\partial g_{\mu\sigma}}{\partial x^\nu} - \frac{\partial g_{\mu\nu}}{\partial x^\sigma} \right)$

**Note:** Computation may start to take noticeable time here.

In [ ]:
# --- Compute and Display Christoffel Symbols ---
#   WARNING: This calculation can take significant time!

print("Computing Christoffel Symbols Γ^ρ_{μν}...")
print("(This may take several minutes...)")

try:
    christoffel_symbols_kerr = compute_christoffel_symbols(coords, metric_kerr)

    # Display the non-zero Christoffel symbols
    # The output will be very large. Displaying is optional.
    if christoffel_symbols_kerr:
        print("\nChristoffel Symbols computed. Displaying non-zero components (output will be long):")
        display_christoffel_symbols(christoffel_symbols_kerr, coords) # Uncomment to display
        print("\n(Displaying all Christoffel symbols is omitted due to length. Calculation finished.)")
    else:
        print("Could not compute Christoffel symbols (check 'gr' module import or calculation error).")
except Exception as e:
    print(f"\nAn error occurred during Christoffel symbol calculation: {e}")
    print("This might be due to complexity or timeouts.")
    christoffel_symbols_kerr = None # Ensure variable exists

## 3. Riemann Curvature Tensor ($R^ρ_{σ μ ν}$)

The Riemann tensor for Kerr spacetime is non-zero, reflecting its curvature. Its components are extremely complicated functions of $r, \theta, M, a$. Calculating them symbolically is a major task.

$R^\rho_{\sigma\mu\nu} = \frac{\partial \Gamma^\rho_{\nu\sigma}}{\partial x^\mu} - \frac{\partial \Gamma^\rho_{\mu\sigma}}{\partial x^\nu} + \Gamma^\rho_{\mu\lambda} \Gamma^\lambda_{\nu\sigma} - \Gamma^\rho_{\nu\lambda} \Gamma^\lambda_{\mu\sigma}$

**Note:** This calculation is computationally very demanding and likely **impractical** to run fully in many environments including Colab free tier. We will attempt it but may skip the display. The primary goal is to compute the Ricci tensor next.

In [ ]:
# --- Compute Riemann Curvature Tensor ---
#   WARNING: This is likely too computationally expensive for typical interactive sessions!
#            It may time out or exhaust resources. Proceed with caution.

print("Computing Riemann Curvature Tensor R^ρ_{σ μ ν}...")
print("!! This calculation is VERY computationally intensive and may fail or take an extremely long time (hours) !!")

# We might skip the actual computation here for practicality in a tutorial setting
# Set flag to attempt computation
attempt_riemann_computation = False # Set to True to attempt, False to skip

riemann_tensor_kerr = None
if attempt_riemann_computation:
    if christoffel_symbols_kerr: # Only proceed if Christoffels were computed
        try:
            riemann_tensor_kerr = compute_riemann_curvature_tensor(coords, metric_kerr, christoffel_symbols=christoffel_symbols_kerr)
            if riemann_tensor_kerr:
                print("\nRiemann tensor computation finished (results likely non-zero but too complex to display).")
                display_riemann_curvature_tensor(riemann_tensor_kerr, coords) # Omitted due to extreme length
            else:
                print("Riemann tensor computation did not return a result.")
        except Exception as e:
            print(f"\nAn error occurred during Riemann tensor calculation: {e}")
            print("This is common due to the extreme complexity.")
    else:
         print("\nSkipping Riemann tensor computation as Christoffel symbols are missing.")
else:
    print("\nSkipping Riemann tensor computation due to expected high computational cost.")

## 4. Ricci Curvature Tensor ($R_{μν}$)

The Ricci tensor is $R_{\mu\nu} = R^\rho_{\mu\rho\nu}$. Since the Kerr metric is a **vacuum** solution ($T_{\mu\nu}=0$), Einstein's equations ($G_{\mu\nu} = 8\pi T_{\mu\nu}$) require $G_{\mu\nu}=0$. For non-cosmological constant spacetimes, $G_{\mu\nu}=0$ implies $R_{\mu\nu}=0$. Therefore, we **must** find that all components of the Ricci tensor are zero after simplification.

This calculation involves contracting the extremely complex Riemann tensor components (or calculating directly, which is also complex) and simplifying the result. Achieving zero relies heavily on the symbolic simplification algorithms.

**Note:** This is the **most critical and computationally intensive verification step**. It may take a very long time.

In [ ]:
# --- Compute and Display Ricci Tensor ---
#   WARNING: This is the most demanding calculation and verification step.

print("Computing Ricci Tensor R_{μν}...")
print("!! This verification step is EXTREMELY computationally intensive and may take a very long time (minutes to hours) !!")

ricci_tensor_kerr = None
# We compute Ricci directly, which might be more feasible than computing Riemann first in some cases
# The gr module computes Ricci tensor from scratch if Riemann tensor is not provided
try:
    ricci_tensor_kerr = compute_ricci_tensor(coords, metric_kerr, riemann_tensor=None) # Force direct calculation or use riemann_tensor_kerr if computed
    # ricci_tensor_kerr = compute_ricci_tensor(coords, metric_kerr, riemann_tensor=riemann_tensor_kerr) # Alternative if Riemann was computed

    if ricci_tensor_kerr:
        print("\nRicci tensor computation finished. Now verifying if all components simplify to zero...")
        # Verification Step: Check if all components simplify to 0
        num_components = len(coords) * len(coords)
        zero_components = 0
        non_zero_components_details = []
        all_zero = True

        for i in range(len(coords)):
            for j in range(len(coords)):
                comp = ricci_tensor_kerr[i][j]
                # Use simplify for verification, allow generous time
                simplified_comp = simplify(comp)
                if simplified_comp != 0:
                    all_zero = False
                    non_zero_components_details.append(f"R_{{{coords[i]},{coords[j]}}} = {simplified_comp}")
                else:
                  zero_components += 1
                # Provide progress update
                print(f"Verifying R_{{{coords[i]},{coords[j]}}}... {'Zero' if simplified_comp == 0 else 'NON-ZERO'}", end='\r')

        print("\nVerification finished.")
        if all_zero:
            print("\nSUCCESS: All components of the Ricci tensor simplified to zero, as expected for a vacuum solution.")
            # Display confirmation using display_ricci_tensor (which shows nothing if all are zero)
            display_ricci_tensor(ricci_tensor_kerr, coords)
        else:
            print("\nFAILURE: Verification failed! Some components of the Ricci tensor did not simplify to zero:")
            print(f"  {zero_components}/{num_components} components verified as zero.")
            print("  Non-zero components found:")
            for detail in non_zero_components_details:
                print(f"    {detail}")
            print("\nThis likely indicates limitations in the symbolic simplification process for this complex metric, or potentially an error in the metric definition/code.")
            # Display the computed (non-simplified or partially simplified) tensor for inspection
            display(Matrix(ricci_tensor_kerr)) # Might be too large

    else:
        print("Ricci tensor computation did not return a result.")

except Exception as e:
    print(f"\nAn error occurred during Ricci tensor calculation or verification: {e}")
    print("This is highly likely due to the extreme complexity and computational demands.")

## 5. Ricci Scalar ($R$)

The Ricci scalar is $R = g^{\mu\nu} R_{\mu\nu}$. Since $R_{\mu\nu}$ must be zero for the Kerr vacuum solution, the Ricci scalar $R$ must also be zero.

In [ ]:
# --- Compute and Display Ricci Scalar ---

print("Computing Ricci Scalar R...")
ricci_scalar_kerr = None

if ricci_tensor_kerr: # Only proceed if Ricci tensor was computed
    try:
        ricci_scalar_kerr = compute_ricci_scalar(coords, metric_kerr, ricci_tensor=ricci_tensor_kerr)

        if ricci_scalar_kerr is not None:
            print("Ricci scalar computation finished. Verifying if it simplifies to zero...")
            # Verification Step
            simplified_scalar = simplify(ricci_scalar_kerr)
            if simplified_scalar == 0:
                print("\nSUCCESS: The Ricci scalar simplified to zero, as expected.")
                display_ricci_scalar(ricci_scalar_kerr)
            else:
                print(f"\nFAILURE: Verification failed! The Ricci scalar did not simplify to zero. Result: {simplified_scalar}")
        else:
            print("Ricci scalar computation did not return a result (potentially due to metric inversion error if Ricci tensor was not exactly zero).")

    except Exception as e:
        print(f"\nAn error occurred during Ricci scalar calculation or verification: {e}")
else:
    print("\nSkipping Ricci scalar computation as Ricci tensor is missing or computation failed.")

## 6. Einstein Tensor ($G_{μν}$)

The Einstein tensor is $G_{\mu\nu} = R_{\mu\nu} - \frac{1}{2} g_{\mu\nu} R$. Since both $R_{\mu\nu}$ and $R$ are expected to be zero for the Kerr vacuum solution, the Einstein tensor $G_{\mu\nu}$ must also be identically zero. This is the ultimate verification of consistency with Einstein's Field Equations $G_{\mu\nu} = 8\pi T_{\mu\nu}$ when $T_{\mu\nu}=0$.

In [ ]:
# --- Compute and Display Einstein Tensor ---

print("Computing Einstein Tensor G_{μν}...")
einstein_tensor_kerr = None

if ricci_tensor_kerr and ricci_scalar_kerr is not None: # Only proceed if prerequisites are available
    # We rely on the fact that R_munu and R *should* be zero.
    # If the previous verification failed, this will likely also show non-zero terms.
    try:
        einstein_tensor_kerr = compute_einstein_tensor(coords, metric_kerr, ricci_tensor=ricci_tensor_kerr, ricci_scalar=ricci_scalar_kerr)

        if einstein_tensor_kerr:
            print("Einstein tensor computation finished. Verifying if all components simplify to zero...")
            # Verification Step
            all_zero_G = True
            non_zero_G_details = []
            num_components_G = len(coords) * len(coords)
            zero_components_G = 0

            for i in range(len(coords)):
                for j in range(len(coords)):
                    comp_G = einstein_tensor_kerr[i][j]
                    simplified_comp_G = simplify(comp_G)
                    if simplified_comp_G != 0:
                        all_zero_G = False
                        non_zero_G_details.append(f"G_{{{coords[i]},{coords[j]}}} = {simplified_comp_G}")
                    else:
                        zero_components_G += 1
                    print(f"Verifying G_{{{coords[i]},{coords[j]}}}... {'Zero' if simplified_comp_G == 0 else 'NON-ZERO'}", end='\r')

            print("\nVerification finished.")
            if all_zero_G:
                print("\nSUCCESS: All components of the Einstein tensor simplified to zero, consistent with a vacuum solution.")
                display_einstein_tensor(einstein_tensor_kerr, coords)
            else:
                print("\nFAILURE: Verification failed! Some components of the Einstein tensor did not simplify to zero:")
                print(f"  {zero_components_G}/{num_components_G} components verified as zero.")
                print("  Non-zero components found:")
                for detail in non_zero_G_details:
                    print(f"    {detail}")
                print("\nThis likely stems from non-zero Ricci components found earlier due to simplification limitations.")

        else:
            print("Einstein tensor computation did not return a result.")

    except Exception as e:
        print(f"\nAn error occurred during Einstein tensor calculation or verification: {e}")
else:
    print("\nSkipping Einstein tensor computation as Ricci tensor or scalar are missing or computation failed.")

## 7. Conclusion

This notebook attempted to compute the geometric tensors for the Kerr metric using the `gr` module and SymPy. Key takeaways:

* The Kerr metric setup in Boyer-Lindquist coordinates involves the mass $M$, spin parameter $a$, and auxiliary functions $\Sigma$ and $\Delta$.
* Key physical features like event horizons and the static limit (ergosphere boundary) were derived from the metric components.
* The Christoffel symbols are complex, reflecting the rotating spacetime.
* The Riemann tensor is non-zero (indicating curvature) but computationally prohibitive to calculate and display fully in most interactive symbolic settings.
* **Crucially, the Kerr metric is a vacuum solution, requiring $R_{\mu\nu}=0$, $R=0$, and $G_{\mu\nu}=0$.**
* Verifying these zero tensors symbolically is a major challenge due to the extreme algebraic complexity. The success depends heavily on the power of the simplification algorithms.

If the verification steps succeeded, it confirms the correctness of the Kerr metric as a vacuum solution within the computational limits. If they failed (resulting in non-zero components after simplification), it most likely highlights the limitations of automated symbolic simplification for such complex expressions, rather than a fundamental flaw in the metric or theory itself (assuming the input metric was correct).

This exercise underscores both the power and the challenges of using symbolic computation in General Relativity, especially for non-trivial solutions like the Kerr metric.